# Kata 21 — Calidad de Descripciones de Tools

> Spec: [`specs/021-tool-descriptions/spec.md`](../../specs/021-tool-descriptions/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

La descripción de un tool es lo único que el modelo lee para decidir cuál llamar. Si dos tools tienen descripciones genéricas similares, el modelo escoge mal silenciosamente.

## 2. Por qué importa

Misrouting de tools es invisible en logs porque el output "se ve razonable". Sólo se detecta downstream cuando algo rompe.

## 3. Caso correcto — descripciones diferenciadas

In [ ]:
import json

# Bueno: descripciones diferenciadas con frontera explícita
TOOLS_GOOD = [
    {
        "name": "extract_web_results",
        "description": (
            "Extrae items de páginas HTML/SERP en formato {title, url, snippet}. "
            "Usa este tool cuando el input es una URL, raw HTML o markdown de "
            "una búsqueda web. Para PDFs o documentos parseados usa parse_document."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"url_or_html": {"type": "string"}},
            "required": ["url_or_html"],
        },
    },
    {
        "name": "parse_document",
        "description": (
            "Extrae texto y estructura de PDFs, DOCX o XLSX. Recibe un path al "
            "archivo binario y devuelve {sections, tables, metadata}. Para HTML "
            "o búsquedas web usa extract_web_results."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"file_path": {"type": "string"}},
            "required": ["file_path"],
        },
    },
]

queries = [
    "Aquí tienes un HTML de Google: <html>...resultados...</html>. ¿Qué resultados saca?",
    "Tengo un PDF en /tmp/factura.pdf. Extrae sus tablas.",
    "Esta página: https://example.com/search?q=foo. Lista resultados.",
]

print("=== descripciones diferenciadas ===")
for q in queries:
    resp = client.messages.create(
        system="Usa la herramienta apropiada para responder.",
        messages=[{"role": "user", "content": q}],
        tools=TOOLS_GOOD,
        tool_choice={"type": "any"},
    )
    tu = next((b for b in resp.content if b.type == "tool_use"), None)
    print(f"  query: {q[:50]}... → tool: {tu.name if tu else 'none'}")

## 4. Anti-patrón — descripciones genéricas

In [ ]:
TOOLS_BAD = [
    {
        "name": "analyze_content",
        "description": "Analyzes content.",
        "input_schema": {"type": "object", "properties": {"input": {"type": "string"}}, "required": ["input"]},
    },
    {
        "name": "analyze_document",
        "description": "Analyzes documents.",
        "input_schema": {"type": "object", "properties": {"input": {"type": "string"}}, "required": ["input"]},
    },
]

print("=== descripciones genéricas ===")
for q in queries:
    resp = client.messages.create(
        system="Usa la herramienta apropiada para responder.",
        messages=[{"role": "user", "content": q}],
        tools=TOOLS_BAD,
        tool_choice={"type": "any"},
    )
    tu = next((b for b in resp.content if b.type == "tool_use"), None)
    print(f"  query: {q[:50]}... → tool: {tu.name if tu else 'none'}")

Con descripciones diferenciadas, las queries de URL/HTML van a `extract_web_results` y la de PDF va a `parse_document`. Con descripciones genéricas, el modelo adivina y rotará entre los dos según signals débiles del prompt — incluso es vulnerable a la presencia de la palabra "documento" en el query.

## 5. Argumento de certificación

- **Descripción = contrato de uso**, no un docstring decorativo.
- **Frontera explícita** ("usa esto en lugar de X cuando…") evita
  misrouting.
- **Renombrar > explicar más** si los nombres son confusos
  (`analyze_content` → `extract_web_results`).
- **Splitting > overloading**: cinco tools con un propósito cada uno
  baten un tool con `mode` parameter.
- Conexión: las descripciones viajan en el system context y se cachean
  (Kata 10) — vale la pena invertir tokens en hacerlas buenas.

## 6. Auto-evaluación

**1. Si dos tools quedan solapados, ¿prefiero renombrar o "explicar más"?**

Renombrar primero. La causa raíz suele ser nombre confuso; descripciones
extras no compensan un mal nombre. Si tras renombrar siguen ambiguos,
splittear funcionalidad.

**2. ¿Cómo mido empíricamente la tasa de tool misrouting?**

Conjunto de queries con ground-truth de qué tool corresponde, ejecutar
y comparar. La celda §3 hace exactamente eso a pequeña escala. En
producción: log de `(query, tool_invoked, expected_tool)` con
expected_tool inferido de la consecuencia downstream.

**3. ¿Qué hago si el system prompt usa una keyword que el modelo asocia
con el tool incorrecto?**

Reescribir el prompt para no usar esa keyword o cambiar el nombre del
tool. El examen ataca este caso con preguntas tipo "el system prompt
dice 'analyze the content' y el tool `analyze_document` se llama por
error" — la fix es evitar la colisión semántica.